# Test element-balanced UniversalXAS selection and fine-tuning

This notebook tests a balanced relative-loss objective for UniversalXAS while holding the unscaled Figshare targets, material splits, and exported 64D features fixed. It trains five UniversalXAS heads with different random seeds. For each element, it selects the candidate checkpoint with the best validation eta, then fine-tunes that checkpoint on the element's training data.

The last section compares the new heads with the existing raw-MSE UniversalXAS and tuned heads from the same feature run. Test metrics are only calculated after validation selection and fine-tuning are complete. The spectra themselves remain unscaled; only the training loss is divided by each element's train-only baseline.

## Remote notebook setup

Run this notebook in the OmniXAS Python environment from the repository or `tutorial_omnixas` directory. If the notebook server stores the repository or data elsewhere, set `OMNIXAS_REPO_ROOT`, `OMNIXAS_DATA_ROOT`, `OMNIXAS_SPLIT_ROOT`, and/or `OMNIXAS_FEATURE_RUN_DIR` in the remote environment before starting Jupyter. The output folder can be changed with `OMNIXAS_HEAD_SWEEP_DIR`.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
repo_env = os.environ.get('OMNIXAS_REPO_ROOT')
if repo_env:
    REPO_ROOT = Path(repo_env).expanduser().resolve()
else:
    candidates = [cwd, *cwd.parents]
    REPO_ROOT = next((p for p in candidates if (p / 'omnixas').is_dir() and (p / 'tutorial_omnixas').is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Cannot find the OmniXAS repository. Set OMNIXAS_REPO_ROOT to its path.')
sys.path.insert(0, str(REPO_ROOT))
DATA_ROOT = Path(os.environ.get('OMNIXAS_DATA_ROOT', REPO_ROOT.parent / 'OmniXAS_data')).expanduser().resolve()
SPLIT_ROOT = Path(os.environ.get('OMNIXAS_SPLIT_ROOT', DATA_ROOT / 'figshare_141_splits_unscaled')).expanduser().resolve()
FEATURE_RUN_DIR = Path(os.environ.get(
    'OMNIXAS_FEATURE_RUN_DIR',
    REPO_ROOT / 'output' / 'training' / 'figshare_141_unscaled_pipeline' / 'figshare_141_unscaled_seed42',
)).expanduser().resolve()
FEATURE_ROOT = FEATURE_RUN_DIR / 'features'
OUTPUT_DIR = Path(os.environ.get(
    'OMNIXAS_HEAD_SWEEP_DIR', FEATURE_RUN_DIR / 'balanced_relative_head_sweep_5seeds',
)).expanduser().resolve()
ELEMENTS = ['Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
SPLITS = ['train', 'val', 'test']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for required in [
    SPLIT_ROOT / 'report.json', SPLIT_ROOT / 'manifest.csv',
    FEATURE_RUN_DIR / 'RUN_COMPLETE.json', FEATURE_RUN_DIR / 'best_encoder.ckpt',
    FEATURE_ROOT,
]:
    if not required.exists():
        raise FileNotFoundError(f'Required remote notebook input is missing: {required}')
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Output already exists: {OUTPUT_DIR}. Set OMNIXAS_HEAD_SWEEP_DIR to a fresh directory to preserve prior results.')

report = json.loads((SPLIT_ROOT / 'report.json').read_text(encoding='utf-8'))
if report.get('intensity_normalization') != 'none (factor 1.0; no xsedge-derived scale)':
    raise ValueError(f'Expected unscaled Figshare targets; found {report.get("intensity_normalization")!r}')
manifest = pd.read_csv(SPLIT_ROOT / 'manifest.csv')
if not np.allclose(pd.to_numeric(manifest['target_scale_factor']), 1.0, rtol=0, atol=0):
    raise ValueError('Split manifest contains a target scale other than 1.0')

# Confirm that exported features, source targets, IDs, and manifest rows agree.
data = {}
for element in ELEMENTS:
    for split in SPLITS:
        task = f'{element}_FEFF'
        ids_path = SPLIT_ROOT / 'material_id_and_site' / f'{task}_{split}.txt'
        y_path = SPLIT_ROOT / 'ml_data' / f'{task}_{split}_y.txt'
        x_feature_path = FEATURE_ROOT / f'{task}_{split}_X.txt'
        y_feature_path = FEATURE_ROOT / f'{task}_{split}_y.txt'
        ids = [line.strip() for line in ids_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
        x = np.atleast_2d(np.loadtxt(x_feature_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        y_feature = np.atleast_2d(np.loadtxt(y_feature_path, dtype=np.float32))
        rows = manifest[(manifest.element == element) & (manifest.split == split)].sort_values('row_index')
        if rows.material_id_and_site.tolist() != ids:
            raise ValueError(f'Manifest/ID order mismatch for {task} {split}')
        if x.shape != (len(ids), 64) or y.shape != (len(ids), 141) or y_feature.shape != y.shape:
            raise ValueError(f'Unexpected feature/target dimensions for {task} {split}: X={x.shape}, y={y.shape}, feature-y={y_feature.shape}')
        if not np.isfinite(x).all() or not np.isfinite(y).all() or (y < 0).any():
            raise ValueError(f'Non-finite or negative values in {task} {split}')
        if not np.allclose(y, y_feature, rtol=1e-6, atol=1e-7):
            raise ValueError(f'Cached features were generated from different targets for {task} {split}')
        data[(element, split)] = {'ids': ids, 'X': x, 'y': y}

# Fixed train-only reference values for the loss; no validation/test values enter training.
TRAIN_MEANS = {}
LOSS_SCALES = {}
for element in ELEMENTS:
    y_train = data[(element, 'train')]['y']
    train_mean = y_train.mean(axis=0, keepdims=True)
    per_spectrum_baseline = np.mean((y_train - train_mean) ** 2, axis=1)
    TRAIN_MEANS[element] = train_mean.astype(np.float32)
    LOSS_SCALES[element] = float(np.median(per_spectrum_baseline))
    if not np.isfinite(LOSS_SCALES[element]) or LOSS_SCALES[element] <= 0:
        raise ValueError(f'Invalid train-only loss scale for {element}: {LOSS_SCALES[element]}')

# Existing checkpoints are the raw-MSE controls from the completed Figshare run.
OLD_UNIVERSAL = FEATURE_RUN_DIR / 'heads' / 'universalXAS' / 'best.pt'
OLD_TUNED = {e: FEATURE_RUN_DIR / 'heads' / 'tunedUniversalXAS' / f'{e}_FEFF' / 'best.pt' for e in ELEMENTS}
missing_heads = [p for p in [OLD_UNIVERSAL, *OLD_TUNED.values()] if not p.is_file()]
if missing_heads:
    raise FileNotFoundError('Completed raw-MSE comparison heads are missing:\n' + '\n'.join(map(str, missing_heads)))

OUTPUT_DIR.mkdir(parents=True)
config = {
    'dataset': str(SPLIT_ROOT),
    'feature_run': str(FEATURE_RUN_DIR),
    'source_encoder_checkpoint': str(FEATURE_RUN_DIR / 'best_encoder.ckpt'),
    'target_policy': report.get('target_policy'),
    'intensity_normalization': report.get('intensity_normalization'),
    'objective': 'mean across 8 elements of per-element batch MSE divided by that element train-only median train-mean baseline MSE',
    'universal_seeds': [42, 43, 44, 45, 46],
    'universal_lr': 5e-4,
    'fine_tune_lr': 3e-4,
    'checkpoint_selection': 'per-element validation eta; test evaluated after selection only',
    'device': str(DEVICE),
}
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
print(f'Repository: {REPO_ROOT}')
print(f'Dataset: {SPLIT_ROOT}')
print(f'Feature run: {FEATURE_RUN_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')
print('Targets are verified unscaled; feature/target/ID rows match for all 24 element-splits.')

## Training setup

Each candidate UniversalXAS head sees a balanced number of training rows per element, sampled with replacement to match the largest element's training count. Within each batch, its loss is the mean of eight element-specific relative MSEs. The denominator for each element is that element's train-only train-mean-spectrum baseline MSE.

For each candidate seed, the notebook saves the best validation checkpoint separately for every element. It then selects the seed/checkpoint with the highest validation eta for each element. No test metric is used for checkpoint or seed selection.

In [ ]:
from omnixas.model.m3gnet_xas import XASSpectralHead

N_UNIVERSAL_SEEDS = [42, 43, 44, 45, 46]
UNIVERSAL_LR = 5e-4
UNIVERSAL_MAX_EPOCHS = 800
UNIVERSAL_PATIENCE = 60
FINE_TUNE_LR = 3e-4
FINE_TUNE_MAX_EPOCHS = 1000
FINE_TUNE_PATIENCE = 60
BATCH_SIZE = 4096


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cpu_state_dict(model):
    return {key: value.detach().cpu() for key, value in model.state_dict().items()}


def evaluate_element(head, element, split):
    item = data[(element, split)]
    x = torch.as_tensor(item['X'], dtype=torch.float32, device=DEVICE)
    y = torch.as_tensor(item['y'], dtype=torch.float32, device=DEVICE)
    head.eval()
    with torch.inference_mode():
        prediction = head(x)
        errors = (prediction - y).square().mean(dim=1).cpu().numpy()
    return errors


def validation_summary(head):
    scores = {}
    for element in ELEMENTS:
        errors = evaluate_element(head, element, 'val')
        median_mse = float(np.median(errors))
        baseline_errors = np.mean((data[(element, 'val')]['y'] - TRAIN_MEANS[element]) ** 2, axis=1)
        baseline_median = float(np.median(baseline_errors))
        scores[element] = {
            'median_mse': median_mse,
            'baseline_median_mse': baseline_median,
            'eta': baseline_median / max(median_mse, 1e-12),
            'relative_median_mse': median_mse / LOSS_SCALES[element],
        }
    return scores


def balanced_training_arrays(seed):
    rng = np.random.default_rng(seed)
    rows_per_element = max(len(data[(element, 'train')]['y']) for element in ELEMENTS)
    xs, ys, task_ids = [], [], []
    for task_index, element in enumerate(ELEMENTS):
        item = data[(element, 'train')]
        selected = rng.integers(0, len(item['y']), size=rows_per_element)
        xs.append(item['X'][selected])
        ys.append(item['y'][selected])
        task_ids.append(np.full(rows_per_element, task_index, dtype=np.int64))
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(task_ids)


def train_universal_candidate(seed):
    set_seed(seed)
    seed_dir = OUTPUT_DIR / 'universal_candidates' / f'seed_{seed}'
    task_ckpt_dir = seed_dir / 'best_by_element'
    task_ckpt_dir.mkdir(parents=True, exist_ok=False)
    train_x, train_y, train_task = balanced_training_arrays(seed)
    train_x = torch.as_tensor(train_x, dtype=torch.float32, device=DEVICE)
    train_y = torch.as_tensor(train_y, dtype=torch.float32, device=DEVICE)
    train_task = torch.as_tensor(train_task, dtype=torch.long, device=DEVICE)
    loss_scale = torch.as_tensor([LOSS_SCALES[e] for e in ELEMENTS], dtype=torch.float32, device=DEVICE)
    head = XASSpectralHead().to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=UNIVERSAL_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=16, min_lr=1e-6)
    best_task_relative = {e: float('inf') for e in ELEMENTS}
    best_macro_relative = float('inf')
    stale = 0
    history = []
    progress = tqdm(range(UNIVERSAL_MAX_EPOCHS), desc=f'UniversalXAS seed {seed}', unit='epoch')
    for epoch in progress:
        head.train()
        order = torch.randperm(len(train_x), device=DEVICE)
        for batch_indices in order.split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            prediction = head(train_x[batch_indices])
            per_row_mse = (prediction - train_y[batch_indices]).square().mean(dim=1)
            batch_tasks = train_task[batch_indices]
            by_element = []
            for task_index in range(len(ELEMENTS)):
                mask = batch_tasks == task_index
                if not mask.any():
                    raise RuntimeError(f'Universal batch lacks {ELEMENTS[task_index]} examples')
                by_element.append(per_row_mse[mask].mean() / loss_scale[task_index])
            loss = torch.stack(by_element).mean()
            loss.backward()
            optimizer.step()

        scores = validation_summary(head)
        macro_relative = float(np.mean([scores[e]['relative_median_mse'] for e in ELEMENTS]))
        history.append({'epoch': epoch + 1, 'macro_relative_median_mse': macro_relative,
                        **{f'{e}_val_eta': scores[e]['eta'] for e in ELEMENTS}})
        for element in ELEMENTS:
            relative = scores[element]['relative_median_mse']
            if relative < best_task_relative[element]:
                best_task_relative[element] = relative
                torch.save({
                    'state_dict': cpu_state_dict(head), 'seed': seed, 'epoch': epoch + 1,
                    'element': element, 'val_eta': scores[element]['eta'],
                    'val_median_mse': scores[element]['median_mse'],
                }, task_ckpt_dir / f'{element}.pt')
        if macro_relative < best_macro_relative:
            best_macro_relative, stale = macro_relative, 0
            torch.save({
                'state_dict': cpu_state_dict(head), 'seed': seed, 'epoch': epoch + 1,
                'macro_relative_median_mse': macro_relative,
            }, seed_dir / 'best_macro.pt')
        else:
            stale += 1
        scheduler.step(macro_relative)
        progress.set_postfix(val_rel=f'{macro_relative:.4f}', best=f'{best_macro_relative:.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.1e}')
        if stale >= UNIVERSAL_PATIENCE:
            break
    progress.close()
    pd.DataFrame(history).to_csv(seed_dir / 'validation_history.csv', index=False)
    records = []
    for element in ELEMENTS:
        checkpoint = task_ckpt_dir / f'{element}.pt'
        saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
        records.append({
            'element': element, 'seed': seed, 'epoch': saved['epoch'],
            'val_median_mse': saved['val_median_mse'], 'val_eta': saved['val_eta'],
            'checkpoint': str(checkpoint),
        })
    return records


universal_val_records = []
for seed in N_UNIVERSAL_SEEDS:
    universal_val_records.extend(train_universal_candidate(seed))
universal_val = pd.DataFrame(universal_val_records)
universal_val.to_csv(OUTPUT_DIR / 'universal_candidate_validation.csv', index=False)
selected_universal = (
    universal_val.sort_values(['element', 'val_eta'], ascending=[True, False])
    .drop_duplicates('element', keep='first').reset_index(drop=True)
)
selected_universal.to_csv(OUTPUT_DIR / 'selected_universal_by_element.csv', index=False)
display(selected_universal[['element', 'seed', 'epoch', 'val_median_mse', 'val_eta']])

## Fine-tune the selected UniversalXAS checkpoint per element

Fine-tuning starts from the validation-selected UniversalXAS checkpoint for that element. It keeps the spectrum targets unchanged and uses train-baseline-relative MSE for optimization. The best fine-tuned checkpoint is selected by validation median MSE (equivalent to validation eta for that fixed element and split).

In [ ]:
def train_tuned_head(element, source_checkpoint, seed):
    set_seed(seed)
    out_dir = OUTPUT_DIR / 'tuned_heads' / f'{element}_FEFF'
    out_dir.mkdir(parents=True, exist_ok=False)
    checkpoint = out_dir / 'best.pt'
    source = torch.load(source_checkpoint, map_location='cpu', weights_only=False)
    head = XASSpectralHead().to(DEVICE)
    head.load_state_dict(source['state_dict'], strict=True)
    optimizer = torch.optim.Adam(head.parameters(), lr=FINE_TUNE_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=16, min_lr=1e-6)
    item_train, item_val = data[(element, 'train')], data[(element, 'val')]
    train_x = torch.as_tensor(item_train['X'], dtype=torch.float32, device=DEVICE)
    train_y = torch.as_tensor(item_train['y'], dtype=torch.float32, device=DEVICE)
    val_x = torch.as_tensor(item_val['X'], dtype=torch.float32, device=DEVICE)
    val_y = torch.as_tensor(item_val['y'], dtype=torch.float32, device=DEVICE)
    scale = LOSS_SCALES[element]
    best_val_relative, stale = float('inf'), 0
    history = []
    progress = tqdm(range(FINE_TUNE_MAX_EPOCHS), desc=f'Tune {element} from seed {source["seed"]}', unit='epoch')
    for epoch in progress:
        head.train()
        order = torch.randperm(len(train_x), device=DEVICE)
        for batch_indices in order.split(BATCH_SIZE):
            optimizer.zero_grad(set_to_none=True)
            loss = (head(train_x[batch_indices]) - train_y[batch_indices]).square().mean() / scale
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.inference_mode():
            val_errors = (head(val_x) - val_y).square().mean(dim=1).cpu().numpy()
        val_median = float(np.median(val_errors))
        relative = val_median / scale
        history.append({'epoch': epoch + 1, 'val_median_mse': val_median, 'val_relative_median_mse': relative})
        if relative < best_val_relative:
            best_val_relative, stale = relative, 0
            val_baseline = float(np.median(np.mean((item_val['y'] - TRAIN_MEANS[element]) ** 2, axis=1)))
            torch.save({
                'state_dict': cpu_state_dict(head), 'seed': seed, 'source_seed': source['seed'],
                'source_epoch': source['epoch'], 'epoch': epoch + 1,
                'val_median_mse': val_median, 'val_eta': val_baseline / max(val_median, 1e-12),
            }, checkpoint)
        else:
            stale += 1
        scheduler.step(relative)
        progress.set_postfix(val_rel=f'{relative:.4f}', best=f'{best_val_relative:.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.1e}')
        if stale >= FINE_TUNE_PATIENCE:
            break
    progress.close()
    pd.DataFrame(history).to_csv(out_dir / 'validation_history.csv', index=False)
    return checkpoint


selected_paths = {row.element: Path(row.checkpoint) for row in selected_universal.itertuples()}
tuned_paths = {}
tuned_val_records = []
for index, element in enumerate(ELEMENTS):
    seed = 10_000 + index
    tuned_paths[element] = train_tuned_head(element, selected_paths[element], seed)
    saved = torch.load(tuned_paths[element], map_location='cpu', weights_only=False)
    tuned_val_records.append({
        'element': element, 'selected_universal_seed': saved['source_seed'],
        'selected_universal_epoch': saved['source_epoch'], 'fine_tune_epoch': saved['epoch'],
        'val_median_mse': saved['val_median_mse'], 'val_eta': saved['val_eta'],
        'checkpoint': str(tuned_paths[element]),
    })
tuned_val = pd.DataFrame(tuned_val_records)
tuned_val.to_csv(OUTPUT_DIR / 'tuned_validation.csv', index=False)
display(tuned_val[['element', 'selected_universal_seed', 'selected_universal_epoch', 'fine_tune_epoch', 'val_median_mse', 'val_eta']])

## Final comparison

This cell evaluates the existing raw-MSE heads and the new balanced-loss heads on validation and test. The test split is first used here, after all seed and checkpoint selections are fixed. Compare the new balanced UniversalXAS and tuned results with the existing heads element by element; do not select another model based on the test table.

In [ ]:
def load_head(checkpoint):
    saved = torch.load(checkpoint, map_location='cpu', weights_only=False)
    model = XASSpectralHead().to(DEVICE)
    model.load_state_dict(saved['state_dict'], strict=True)
    return model.eval()


def score_on_split(element, variant, split, checkpoint):
    head = load_head(checkpoint)
    errors = evaluate_element(head, element, split)
    y = data[(element, split)]['y']
    baseline_errors = np.mean((y - TRAIN_MEANS[element]) ** 2, axis=1)
    baseline_median = float(np.median(baseline_errors))
    median_mse = float(np.median(errors))
    return {
        'element': element, 'variant': variant, 'evaluation_split': split,
        'n_spectra': len(errors), 'aggregate_mse': float(errors.mean()),
        'median_spectrum_mse': median_mse, 'baseline_median_mse': baseline_median,
        'eta': baseline_median / max(median_mse, 1e-12),
    }


comparison_rows = []
for element in ELEMENTS:
    candidates = [
        ('Existing UniversalXAS', OLD_UNIVERSAL),
        ('Existing Tuned-UniversalXAS', OLD_TUNED[element]),
        ('Balanced UniversalXAS (val-selected)', selected_paths[element]),
        ('Balanced Tuned-UniversalXAS', tuned_paths[element]),
    ]
    for variant, checkpoint in candidates:
        for split in ('val', 'test'):
            comparison_rows.append(score_on_split(element, variant, split, checkpoint))
comparison = pd.DataFrame(comparison_rows).sort_values(['element', 'variant', 'evaluation_split'])
comparison.to_csv(OUTPUT_DIR / 'comparison_validation_and_test.csv', index=False)
test_table = comparison[comparison.evaluation_split == 'test'].copy()
test_table.to_csv(OUTPUT_DIR / 'comparison_test_only.csv', index=False)
display(comparison.round({'aggregate_mse': 6, 'median_spectrum_mse': 6, 'baseline_median_mse': 6, 'eta': 3}))

import matplotlib.pyplot as plt
eta_plot = test_table.pivot(index='element', columns='variant', values='eta').reindex(ELEMENTS)
ax = eta_plot.plot(kind='bar', figsize=(13, 5), rot=0)
ax.set_ylabel('Test eta (higher is better)')
ax.set_title('Existing raw-MSE heads vs validation-selected balanced-loss heads')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'test_eta_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Head sweep checkpoints, validation selection, test metrics, and plot saved in {OUTPUT_DIR}')